# Animate hybridisation terms in the complex plane
Sweeps the frequency across the mechanical resonances and follows each term of the deranged determinant of a 2×2 system (orders 2, 3 and 4, sieved to the distinct terms) and the 4th-order terms of a 5×1 system with the j-sequence x₁x₂x₃x₄, as arrows in the complex plane.

Runtime: about 5-30 s by default, which shows the last frame of each sweep as a static figure. Set `MAKE_ANIMATIONS = True` to write the animations to `output/` (about 2-3 min).

# START

# Generate animations

In [ ]:
from pathlib import Path
import matplotlib
import matplotlib.animation
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt

from levisim import *

set_plt_rcParams_to_xmgrace_style()

# --- run settings ---------------------------------------------------------------------------------
# False (default): every animation is stepped through its frames without being rendered and shown as
# one static figure of its last frame (with every trail), which takes seconds.
# True: every animation is also written to output/ (GIF, or MP4 when ffmpeg is installed), which
# takes minutes. output/ is not tracked by git.
MAKE_ANIMATIONS = False
# keep every n-th frame; n > 1 gives a quick test of the full run
FRAME_STRIDE = 1
# resolution of the written animation files
ANIMATION_DPI = 75
OUTPUT_DIR = Path('output')


class SweepAnimation:
    """
    A frequency-sweep animation, update(frame) over num_frames frames, rendered only on request

    show(name) writes it to OUTPUT_DIR when MAKE_ANIMATIONS is True, otherwise it just runs
    update over the frames; either way it then displays the figure, which holds the last frame.
    """
    def __init__(self, fig, update, num_frames, fps=20):
        self.fig, self.update, self.fps = fig, update, fps
        self.frames = list(range(0, num_frames, FRAME_STRIDE))
        # closed so that the notebook does not also show the empty figure when the cell ends
        plt.close(fig)

    def show(self, name):
        if MAKE_ANIMATIONS:
            OUTPUT_DIR.mkdir(exist_ok=True)
            writer = 'ffmpeg' if matplotlib.animation.writers.is_available('ffmpeg') else 'pillow'
            path = OUTPUT_DIR / f'{name}.{"mp4" if writer == "ffmpeg" else "gif"}'
            animation = matplotlib.animation.FuncAnimation(self.fig, self.update, frames=self.frames, blit=False)
            animation.save(path, writer=writer, fps=self.fps, dpi=ANIMATION_DPI)
            print(f'wrote {path} ({len(self.frames)} frames)')
        else:
            for frame in self.frames:
                self.update(frame)
        self.fig.canvas.draw()
        display(self.fig)


def generate_animation_of_hybridisation_terms(mtrap, terms, num_terms=None, lim=3):
    """
    Generates a `FuncAnimation` for a given series of hybridisation terms
    """
    ω_input = np.linspace(275, 325, 100, endpoint=True) * kHz
    fig, ax = plt.subplots(figsize=(LATEX_FIG_WIDTH, LATEX_FIG_WIDTH), dpi=150)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal', 'box')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.grid(alpha=0.1)

    marker = 'o'
    colours = colours_50
    arrows = [
        ax.plot([], [], color=colour, marker=marker)
        for colour in colours
    ]
    arrows = sum(arrows, [])

    output_trails = [
        ax.scatter([], [], color=colour, alpha=0.5, s=10)
        for colour in colours
    ]
    trails = [[] for colour in colours]

    if num_terms is None:
        num_terms = len(terms)
    arrows = arrows[:num_terms]
    output_trails = output_trails[:num_terms]
    trails = trails[:num_terms]
    terms = terms[:num_terms]

    def update(frame):
        """
        Update the animation for the given frame
        """
        ω = ω_input[frame]

        for ii, term in enumerate(terms):
            fω = mtrap.hybridisation_term_value(term, ω)
            # arrows[ii].set_label(f'{term_string(term)}(ω)={fω:.3f}')
            arrows[ii].set_label(f'${term_latex(term)}=${fω:.3f}')
            arrows[ii].set_data([0, fω.real], [0, fω.imag])
            trails[ii].append((fω.real, fω.imag))
            output_trails[ii].set_offsets(trails[ii])

        ax.legend(loc='lower left', prop=dict(size=8))
        ax.set_title(f'$\omega = ${ω / kHz:.1f} kHz')

        return (*arrows, *output_trails)

    return SweepAnimation(fig, update, len(ω_input))

N, d = 2, 2
mtrap = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=d,
    gas_kwargs=DEFAULT_gas_kwargs,
    cavity_kwargs=DEFAULT_cavity_kwargs,
    toy_params=FORTRAN_toy_params,
    generate_power_series=True,
)

det_2x2_2nd_order = sieve_power_series(mtrap.deranged_determinant_by_order[2], 2)
print(f'{len(det_2x2_2nd_order)=}')
print(f'{det_2x2_2nd_order=}')
print(f'{[term_string(term) for term in det_2x2_2nd_order]=}')
ani_2x2_2nd_order = generate_animation_of_hybridisation_terms(mtrap, det_2x2_2nd_order)

det_2x2_3rd_order = sieve_power_series(mtrap.deranged_determinant_by_order[3], 3)
print(f'{len(det_2x2_3rd_order)=}')
print(f'{det_2x2_3rd_order=}')
print(f'{[term_string(term) for term in det_2x2_3rd_order]=}')
ani_2x2_3rd_order = generate_animation_of_hybridisation_terms(mtrap, det_2x2_3rd_order)

det_2x2_4th_order = sieve_power_series(mtrap.deranged_determinant_by_order[4], 4)
print(f'{len(det_2x2_4th_order)=}')
print(f'{det_2x2_4th_order=}')
print(f'{[term_string(term) for term in det_2x2_4th_order]=}')
ani_2x2_4th_order = generate_animation_of_hybridisation_terms(mtrap, det_2x2_4th_order)

N, d = 5, 1
mtrap = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=d,
    gas_kwargs=DEFAULT_gas_kwargs,
    cavity_kwargs=DEFAULT_cavity_kwargs,
    toy_params=FORTRAN_toy_params,
    generate_power_series=True,
)

sieve_sequence = ['x1', 'x2', 'x3', 'x4']
terms = sieve_power_series_for_sequence(mtrap.deranged_determinant_by_order[4], sieve_sequence)
print(f'{len(terms)=}')
print(f'{terms=}')
print(f'{[term_string(term) for term in terms]=}')
ani_5x1_sieved_for_sequence = generate_animation_of_hybridisation_terms(mtrap, terms)


# 2×2 system

## 2nd order (6 terms)

In [ ]:
ani_2x2_2nd_order.show('2x2_2nd_order')


## 3rd order (8 terms)

In [ ]:
ani_2x2_3rd_order.show('2x2_3rd_order')


## 4th order (9 terms)
- Since the sequence of $j$'s determines the distinctive (unordered) product of $\mu_j(\omega)$ of each hybridisation term, all terms of full order are equivalent up to the ±1 sign
- This therefore means there is a small amount of cancellation that occurs (in equal coupling, no phase case)

In [ ]:
ani_2x2_4th_order.show('2x2_4th_order')


# 5×1 systems: sieved for terms with $j$ sequence $x_1 x_2 x_3 x_4$ (9 terms)
- initial index $j$ determines the $\mu_j(\omega)$ function at the start of each $R_{jk}(\omega)$
- therefore sequences of $j$'s will be equivalent (up to the number-of-cycles-determined sign) if all couplings are equal (and no tweezer phase differences)

In [ ]:
ani_5x1_sieved_for_sequence.show('5x1_sieved_for_sequence')


# END